# 02 — Analyse métier sur les marts validés

**Objectif.** Répondre aux six questions analytiques du projet à partir des **marts validés par le quality gate**.

**Garde-fous.** Ce notebook refuse de s'exécuter si le quality gate n'est pas `PASS`. Il n'applique aucune
transformation : les agrégations vivent dans `src/france_business_dynamics/analysis/queries.py` (testées).

**Lecture prudente.** Les *cessations* et *fermetures administratives* sont des changements d'état dans SIRENE, pas une
« mortalité d'entreprise ». Les trois séries d'événements ne s'additionnent jamais (voir `docs/methodology.md`).

In [ ]:

import matplotlib.pyplot as plt
import polars as pl

from france_business_dynamics.utils.config import SOURCE_FILES, ProjectPaths

pl.Config.set_tbl_rows(25)
pl.Config.set_fmt_str_lengths(60)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

PATHS = ProjectPaths.from_env()
print("Répertoire de données :", PATHS.data)

from france_business_dynamics.analysis import queries as q
from france_business_dynamics.utils.jsonio import read_json

report_path = PATHS.processed / "quality_report.json"
if not report_path.exists():
    raise RuntimeError("quality_report.json absent : exécuter `python -m france_business_dynamics.quality_gate` avant l'analyse.")
report = read_json(report_path)
if report["status"] != "PASS":
    raise RuntimeError("Le quality gate est en échec : corriger les marts avant toute analyse (voir notebook 03).")
if report["contains_samples"]:
    print("⚠ ATTENTION : certaines couches sont des ÉCHANTILLONS —", ", ".join(report["summary"]["sample_layers"]),
          "\n  Les chiffres ci-dessous ne représentent pas la France entière.")
if not report["complete"]:
    print("⚠ Couches non contrôlées :", ", ".join(report["summary"]["layers_skipped"]))

marts = q.load_marts(PATHS)
print("Marts chargés :", ", ".join(marts))
display(pl.DataFrame({"indicateur": list(q.headline_figures(marts)), "valeur": list(q.headline_figures(marts).values())}))

def barh(frame, label, value, title, color="#2c3e50", top=None):
    data = frame.head(top) if top else frame
    plt.barh(data[label].to_list()[::-1], data[value].to_list()[::-1], color=color)
    plt.title(title); plt.tight_layout(); plt.show()

## Q1 — Comment le tissu des établissements évolue-t-il entre 2021 et 2025 ?

> **Ce graphique montre des FLUX administratifs, pas un stock reconstruit.** On y lit, année par année : les
> établissements dont la date de création *enregistrée* tombe dans l'année, les fermetures administratives et les
> cessations d'unités légales. Le nombre d'établissements actifs à chaque fin d'année n'est **pas** calculé ici
> (voir `docs/roadmap.md`, première évolution prévue). Les unités purgées (colonne `unitePurgeeUniteLegale`) pourraient être mal couvertes :
> les années anciennes peuvent être sous-estimées (biais de survie possible, à vérifier dans la documentation Insee).

In [ ]:
if "events" in marts:
    flows = q.events_by_year(marts["events"])
    display(flows)
    years = flows["event_year"].to_list()
    width = 0.27
    for i, (column, color) in enumerate((("Date de création enregistrée", "#27ae60"),
                                         ("Fermeture administrative", "#c0392b"),
                                         ("Cessation administrative", "#7f8c8d"))):
        plt.bar([y + (i - 1) * width for y in years], flows[column].to_list(), width, label=column, color=color)
    plt.xticks(years); plt.legend(); plt.title("Flux annuels (trois séries distinctes — à ne pas additionner)")
    plt.show()
else:
    print("Mart des événements absent : construire avec `build_mart` (sans --skip-events).")

## Q2 — Répartition par région, département et commune

In [ ]:
est = marts.get("establishments")
if est is not None:
    regions = q.establishments_by_region(est)
    display(regions)
    barh(regions, "region", "etablissements", "Établissements actifs par région")
    departments = q.establishments_by_department(est, top_n=15)
    barh(departments, "departement", "etablissements", "Top 15 départements (actifs)", "#16a085")
    communes = q.top_communes(est, top_n=15)
    display(communes)
    print("Qualité de l'appariement géographique :")
    display(q.geography_coverage(est))

## Q3 — Secteurs d'activité (sections NAF Rev.2)

Seuls les établissements en **NAFRev2** reçoivent une section. La ligne « — / Hors NAF Rev.2 » regroupe les
nomenclatures historiques (NAFRev1, NAF1993, NAP) et les codes absents : elle n'est jamais répartie dans les sections.

In [ ]:
if est is not None:
    sections = q.establishments_by_naf_section(est)
    display(sections)
    barh(sections, "secteur", "etablissements", "Établissements actifs par section NAF Rev.2", "#8e44ad")
    print("Couverture des nomenclatures (établissements actifs) :")
    display(q.naf_nomenclature_coverage(est))

## Q4 — Structure par tranche d'effectifs

In [ ]:
if est is not None:
    workforce = q.establishments_by_workforce(est)
    display(workforce)
    plt.bar(workforce["workforce_code"].to_list(), workforce["etablissements"].to_list(), color="#e67e22")
    plt.title("Établissements actifs par tranche d'effectifs (code officiel, ordre croissant)"); plt.show()

## Q5 — Changements administratifs d'état

Cessations d'unités légales et fermetures d'établissements, par année et par secteur. *Une cessation d'unité légale
entraîne des fermetures d'établissements : ne jamais additionner les deux séries.*

In [ ]:
ev = marts.get("events")
if ev is not None:
    flows = q.events_by_year(ev)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(flows["event_year"], flows["Cessation administrative"], marker="o", color="#7f8c8d")
    axes[0].set_title("Cessations administratives (unités légales)")
    axes[1].plot(flows["event_year"], flows["Fermeture administrative"], marker="o", color="#c0392b")
    axes[1].set_title("Fermetures administratives (établissements)")
    for ax in axes: ax.set_xticks(flows["event_year"].to_list())
    plt.tight_layout(); plt.show()
    display(q.events_by_section(ev, "Fermeture administrative").head(10))

## Q6 — Catégories juridiques et secteurs d'activité des unités légales

In [ ]:
lu = marts.get("legal_units")
if lu is not None:
    legal = q.legal_units_by_legal_category(lu)
    display(legal)
    barh(legal, "categorie_juridique", "unites_legales", "Unités légales actives par catégorie juridique (niveau I)", "#2980b9")
    display(q.legal_units_by_enterprise_category(lu))
    display(q.legal_units_by_section(lu).head(10))

## Limites de lecture

* Instantané du stock SIRENE à la date de snapshot du projet ; pas une série historique de stocks.
* Les sections NAF ne couvrent que la NAF Rev.2 ; la NAF 2025 n'est pas exploitée comme dimension d'analyse.
* La tranche d'effectifs renvoie à une année de référence (`anneeEffectifs…`) qui peut être ancienne.
* Détail : `docs/methodology.md`.